# The Math Behind Factor Models — Interactive Guide

You read risk reports every day. You see beta, R-squared, factor exposures, tracking error. You know what the numbers roughly mean, but you're not sure **why** the math works the way it does.

This notebook fixes that. Every concept comes with a runnable example you can **change the inputs** and immediately see how the output shifts. Tweak the numbers, break things, build intuition.

> **How to use this:** Run each cell in order. Cells marked **TRY IT** have variables you should change. Cells marked **EXERCISE** are for you to write code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

TRADING_DAYS = 252
np.random.seed(42)
print('Ready.')

---
## 1. What Is a Factor Model?

Think of factors as **ingredients in a recipe**. A stock's daily return is a mix of:

- How the broad market moved (market factor)
- Whether small-caps outperformed large-caps that day (size factor)
- Whether value stocks beat growth stocks (value factor)
- Something unique to that stock — an earnings surprise, a CEO tweet

The factor model separates these ingredients so you can see which ones are driving your portfolio.

> **Key insight:** If factors explain 80% of a stock's movement, only 20% is genuinely stock-specific. That 20% is what PMs are being paid to find. Investors don't pay 2-and-20 for factor exposure they could get from a Vanguard ETF.

Let's see this decomposition visually.

In [ ]:
# === RETURN DECOMPOSITION — See how a stock's return breaks into pieces ===

np.random.seed(42)
days = 60
dates = pd.date_range('2024-01-01', periods=days, freq='B')

# Simulate factor returns
market_return = np.random.normal(0.0004, 0.012, days)  # SPY-like
smb_return    = np.random.normal(0.0001, 0.005, days)  # size factor
hml_return    = np.random.normal(0.0001, 0.004, days)  # value factor

# Stock's factor exposures (loadings)
beta_mkt = 1.2    # amplifies market moves by 20%
beta_smb = -0.3   # large-cap tilt (moves opposite to small-cap factor)
beta_hml = -0.5   # growth tilt (moves opposite to value factor)
alpha_daily = 0.0003  # small daily alpha (~7.5% annual)

# Decomposed pieces
mkt_piece = beta_mkt * market_return
smb_piece = beta_smb * smb_return
hml_piece = beta_hml * hml_return
noise = np.random.normal(0, 0.008, days)  # idiosyncratic noise

total_return = alpha_daily + mkt_piece + smb_piece + hml_piece + noise

# Stacked bar chart showing decomposition
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Top: total return
colors_pos = np.where(total_return >= 0, 'steelblue', 'salmon')
ax1.bar(range(days), total_return * 100, color=colors_pos, alpha=0.7, width=0.8)
ax1.set_ylabel('Daily Return (%)')
ax1.set_title('Total Stock Return — What You See on the PnL Report')
ax1.axhline(0, color='black', linewidth=0.5)

# Bottom: decomposition
ax2.bar(range(days), mkt_piece * 100, label=f'Market (beta={beta_mkt})', alpha=0.8, color='steelblue')
ax2.bar(range(days), smb_piece * 100, bottom=mkt_piece * 100, label=f'SMB ({beta_smb})', alpha=0.8, color='orange')
ax2.bar(range(days), hml_piece * 100, bottom=(mkt_piece + smb_piece) * 100, label=f'HML ({beta_hml})', alpha=0.8, color='green')
ax2.bar(range(days), noise * 100, bottom=(mkt_piece + smb_piece + hml_piece) * 100, label='Idiosyncratic', alpha=0.5, color='grey')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_ylabel('Return Contribution (%)')
ax2.set_xlabel('Trading Day')
ax2.set_title('Decomposed — What\'s Actually Driving the Return')
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

# Variance attribution
var_total = np.var(total_return)
var_mkt = np.var(mkt_piece)
var_factors = np.var(mkt_piece + smb_piece + hml_piece)
print(f'Market alone explains:     {var_mkt/var_total:.0%} of total variance')
print(f'All 3 factors explain:     {var_factors/var_total:.0%} of total variance')
print(f'Stock-specific (residual): {1 - var_factors/var_total:.0%}')

---
## 2. Beta — Your Stock's Sensitivity to the Market

**Plain English:** If SPY goes up 1% and your stock goes up 1.3%, your stock's beta is approximately **1.3**. It amplifies market moves by 30%.

The equation:

$$R_{\text{stock}} = \alpha + \beta \times R_{\text{market}} + \epsilon$$

| Piece | What it means |
|-------|---------------|
| $R_{\text{stock}}$ | Your stock's return on a given day |
| $\alpha$ | The intercept — return not explained by the market |
| $\beta$ | How much the stock moves per 1% market move |
| $\epsilon$ | Random noise — earnings surprises, stock-specific events |

The scatter plot below makes this tangible. Each dot is a trading day. The **slope of the red line is beta**.

In [ ]:
# === TRY IT: Change 'true_beta' and see how the scatter changes ===

np.random.seed(42)
n = 500  # trading days
market = np.random.normal(0.0005, 0.012, n)  # SPY-like returns

true_beta = 1.3   # <-- CHANGE THIS: try 0.5, 1.0, 1.5, 2.0, -0.3
true_alpha = 0.0002  # daily alpha
noise_vol = 0.008

stock = true_alpha + true_beta * market + np.random.normal(0, noise_vol, n)

# Fit regression line
slope, intercept = np.polyfit(market, stock, 1)
r_squared = np.corrcoef(market, stock)[0, 1] ** 2

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(market * 100, stock * 100, alpha=0.15, s=10, color='steelblue')
x_line = np.linspace(market.min(), market.max(), 100)
ax.plot(x_line * 100, (slope * x_line + intercept) * 100, 'r-', linewidth=2,
        label=f'Estimated beta = {slope:.2f} (true = {true_beta})')
ax.axhline(0, color='grey', linewidth=0.5)
ax.axvline(0, color='grey', linewidth=0.5)
ax.set_xlabel('Market Return (%)')
ax.set_ylabel('Stock Return (%)')
ax.set_title(f'Beta = Slope of the Line  |  R\u00b2 = {r_squared:.1%}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'True beta:      {true_beta}')
print(f'Estimated beta: {slope:.3f}')
print(f'R-squared:      {r_squared:.1%} of stock variance explained by the market')

In [ ]:
# === SIDE-BY-SIDE: What different betas look like ===

np.random.seed(42)
betas = [0.5, 1.0, 1.5]
labels = ['Defensive (0.5)', 'Market-Neutral (1.0)', 'Aggressive (1.5)']
colors = ['green', 'steelblue', 'red']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, b, label, c in zip(axes, betas, labels, colors):
    stock_sim = b * market + np.random.normal(0, 0.006, n)
    ax.scatter(market * 100, stock_sim * 100, alpha=0.15, s=8, color=c)
    s, i = np.polyfit(market, stock_sim, 1)
    x_l = np.linspace(market.min(), market.max(), 50)
    ax.plot(x_l * 100, (s * x_l + i) * 100, 'k-', linewidth=1.5)
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.axvline(0, color='grey', linewidth=0.5)
    ax.set_xlabel('Market Return (%)')
    ax.set_title(f'{label}\nbeta = {s:.2f}', fontsize=11)

axes[0].set_ylabel('Stock Return (%)')
plt.suptitle('Beta Controls the Slope — Same Market, Different Sensitivities', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print('Quick Beta Reference:')
print('  > 1.5  : Very aggressive — high-growth tech, biotech')
print('  ~ 1.0  : Moves with the market — broad index, diversified large-caps')
print('  0.5-0.8: Defensive — utilities, consumer staples')
print('  ~ 0    : Uncorrelated — market-neutral strategies')
print('  < 0    : Moves opposite to market — very rare, some tail hedges')

### What Beta Is NOT

Beta measures **systematic risk** — sensitivity to the broad market. It does NOT measure total risk. A biotech stock waiting on an FDA ruling might have beta = 0.3 (low market sensitivity) but be extraordinarily risky because of a binary event. Beta only captures the market-driven part of risk.

### Why PMs Have Beta Targets

Your desk monitors beta drift because it's **unintended risk**. If a PM has a mandate to run at 0.5 beta and they drift to 1.2, the portfolio is taking on market risk the investor didn't sign up for. That's a risk conversation, not an alpha conversation.

---
## 3. Alpha — The Holy Grail

Alpha is the return left over after you remove all factor exposure. In the equation, it's the **intercept** — the return your PM generated that the market didn't hand them for free.

**Here's the uncomfortable truth:** most "alpha" isn't really alpha. It's hidden factor exposure.

Consider a PM who returned 18% last year:
- Portfolio beta = 1.3, market was up 12% → beta contributed ~15.6%
- Heavily tilted to small-cap value → SMB and HML added another ~3%
- **True alpha: -0.6%**

They didn't pick great stocks. They loaded up on factors that happened to do well. An ETF portfolio with the same tilts would have done better — for 0.03% instead of 2%.

In [ ]:
# === WORKED EXAMPLE: Decomposing a PM's 12% return ===
#
# TRY IT: Change the factor returns or the PM's loadings
# and see how the decomposition shifts.

# Factor returns for the year (what the market/factors delivered)
factor_returns = {
    'MKT': 0.10,   # <-- market returned 10%
    'SMB': 0.05,   # <-- small-caps beat large-caps by 5%
    'HML': 0.03,   # <-- value beat growth by 3%
}

# The PM's factor loadings (exposures)
pm_loadings = {
    'MKT': 1.0,    # <-- market beta of 1.0
    'SMB': 0.3,    # <-- tilted toward small-caps
    'HML': 0.4,    # <-- tilted toward value
}

pm_total_return = 0.12  # The PM's actual return

# Decompose
contributions = {}
for factor in factor_returns:
    contributions[factor] = pm_loadings[factor] * factor_returns[factor]

factor_total = sum(contributions.values())
alpha = pm_total_return - factor_total

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: waterfall chart
labels = list(contributions.keys()) + ['Alpha', 'Total']
values = list(contributions.values()) + [alpha, pm_total_return]
bottoms = [0] * len(contributions)
running = 0
for i, v in enumerate(list(contributions.values())):
    bottoms[i] = running
    running += v
bottoms.append(running)  # alpha starts where factors end
bottoms.append(0)        # total starts at zero

bar_colors = ['steelblue', 'orange', 'green',
              'red' if alpha < 0 else 'gold', 'purple']

ax1.bar(labels, values, bottom=bottoms, color=bar_colors, edgecolor='white', width=0.6)
for i, (v, b) in enumerate(zip(values, bottoms)):
    ax1.text(i, b + v/2, f'{v:.1%}', ha='center', va='center', fontweight='bold', fontsize=10)
ax1.set_title('Return Decomposition (Waterfall)', fontsize=12)
ax1.set_ylabel('Return')
ax1.axhline(0, color='black', linewidth=0.5)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Right: pie chart of what's factor vs alpha
sizes = [abs(factor_total), abs(alpha)]
pie_labels = [f'Factor exposure\n{factor_total:.1%}', f'Alpha\n{alpha:.1%}']
pie_colors = ['steelblue', 'red' if alpha < 0 else 'gold']
ax2.pie(sizes, labels=pie_labels, colors=pie_colors, autopct='%1.0f%%',
        startangle=90, textprops={'fontsize': 11})
ax2.set_title(f'PM returned {pm_total_return:.0%} — how much was skill?', fontsize=12)

plt.tight_layout()
plt.show()

verdict = 'DESTROYED value' if alpha < 0 else 'generated genuine alpha'
print(f'Factor exposure contributed: {factor_total:.1%}')
print(f'True alpha:                 {alpha:.1%}')
print(f'Verdict: The PM {verdict} after accounting for factor risk.')

### Is the Alpha Real? The t-Statistic

Finding alpha in a regression doesn't mean it's real. Markets are noisy. You need to ask: **how confident are we this isn't just luck?**

The t-statistic answers this: "How many standard errors is the alpha away from zero?" If it's far from zero, it's unlikely to be noise.

| t-stat | Interpretation |
|--------|---------------|
| < 1.0 | Could easily be noise. No evidence of real alpha. |
| 1.0-2.0 | Suggestive but not convincing. |
| > 2.0 | Statistically significant at ~95% confidence. |
| > 3.0 | Very strong evidence. Rare in live portfolios. |

An alpha of 3% with t-stat 0.8 is **less impressive** than alpha of 1% with t-stat 2.5. The first is probably noise with a big number attached. The second is small but probably real.

In [ ]:
# === DEMO: Why t-stats matter more than raw alpha ===
#
# We simulate 1000 "PMs" with ZERO true alpha.
# Some of them will LOOK like they have huge alpha — just by luck.

np.random.seed(42)
n_sims = 1000
n_days = 500
market_sim = np.random.normal(0.0004, 0.012, n_days)

alphas = []
tstats = []

for _ in range(n_sims):
    # Each PM has beta=1 and NO true alpha
    stock_sim = 1.0 * market_sim + np.random.normal(0, 0.01, n_days)
    # Regress
    X_sim = np.column_stack([np.ones(n_days), market_sim])
    beta_hat = np.linalg.lstsq(X_sim, stock_sim, rcond=None)[0]
    resid = stock_sim - X_sim @ beta_hat
    se_alpha = np.sqrt(np.var(resid) * np.linalg.inv(X_sim.T @ X_sim)[0, 0])
    alphas.append(beta_hat[0] * TRADING_DAYS)  # annualize
    tstats.append(beta_hat[0] / se_alpha)

alphas = np.array(alphas)
tstats = np.array(tstats)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution of estimated alphas
ax1.hist(alphas * 100, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax1.axvline(0, color='red', linewidth=2, label='True alpha = 0')
ax1.set_xlabel('Estimated Annual Alpha (%)')
ax1.set_ylabel('Count')
ax1.set_title(f'1000 PMs with ZERO true alpha\n{(np.abs(alphas) > 0.03).sum()} of them "found" |alpha| > 3%')
ax1.legend()

# Right: t-stats — most are near zero, but some cross 2
ax2.hist(tstats, bins=50, color='orange', alpha=0.7, edgecolor='white')
ax2.axvline(2, color='red', linewidth=2, linestyle='--', label='t = 2 (significant)')
ax2.axvline(-2, color='red', linewidth=2, linestyle='--')
pct_sig = (np.abs(tstats) > 2).mean()
ax2.set_xlabel('t-statistic')
ax2.set_ylabel('Count')
ax2.set_title(f't-Statistics of Zero-Alpha PMs\n{pct_sig:.0%} appear "significant" by chance alone')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Even with ZERO true alpha, {pct_sig:.0%} of PMs cross the t=2 threshold.')
print('This is why one regression is never enough — you need economic reasoning too.')

---
## 4. Fama-French: Beyond Just the Market

CAPM uses one factor: the market. Fama and French showed that two additional factors explain a large chunk of what CAPM misses.

| Factor | What it captures | How to think about it |
|--------|-----------------|----------------------|
| **MKT** | Market excess return | Same as CAPM beta |
| **SMB** (Small Minus Big) | Size premium | Small-caps vs large-caps |
| **HML** (High Minus Low) | Value premium | Cheap stocks vs expensive stocks |

The regression becomes:

$$R_{\text{stock}} - R_f = \alpha + \beta_1 \times \text{MKT} + \beta_2 \times \text{SMB} + \beta_3 \times \text{HML} + \epsilon$$

Each coefficient tells you how exposed the portfolio is to that factor. Together, they paint a picture of *what kind* of risk the PM is taking.

In [ ]:
# === TRY IT: Change the loadings to see different stock "personalities" ===
#
# Preset profiles (uncomment one, or make your own):
#   Tech growth stock:  beta_mkt=1.3, beta_smb=-0.3, beta_hml=-0.5
#   Regional bank:      beta_mkt=1.1, beta_smb=0.4,  beta_hml=0.6
#   Utility company:    beta_mkt=0.5, beta_smb=-0.1, beta_hml=0.2
#   Market-neutral HF:  beta_mkt=0.05, beta_smb=0.0, beta_hml=0.0

beta_mkt = 1.3     # <-- CHANGE: market sensitivity
beta_smb = -0.3    # <-- CHANGE: size tilt (+ = small-cap, - = large-cap)
beta_hml = -0.5    # <-- CHANGE: value tilt (+ = value, - = growth)

np.random.seed(7)
n = 500

# Simulate factor returns
mkt = np.random.normal(0.0004, 0.012, n)
smb = np.random.normal(0.0001, 0.006, n)
hml = np.random.normal(0.0001, 0.005, n)

# Stock return = factor loadings * factor returns + noise
stock_ret = 0.0002 + beta_mkt * mkt + beta_smb * smb + beta_hml * hml + np.random.normal(0, 0.007, n)

# Run 3-factor regression to recover the loadings
X_3f = np.column_stack([np.ones(n), mkt, smb, hml])
betas_hat = np.linalg.lstsq(X_3f, stock_ret, rcond=None)[0]
y_hat = X_3f @ betas_hat
ss_res = np.sum((stock_ret - y_hat)**2)
ss_tot = np.sum((stock_ret - stock_ret.mean())**2)
r2 = 1 - ss_res / ss_tot

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of loadings
factor_names = ['MKT', 'SMB', 'HML']
true_vals = [beta_mkt, beta_smb, beta_hml]
est_vals = [betas_hat[1], betas_hat[2], betas_hat[3]]

x_pos = np.arange(len(factor_names))
w = 0.35
ax1.bar(x_pos - w/2, true_vals, w, label='True', color='steelblue', edgecolor='white')
ax1.bar(x_pos + w/2, est_vals, w, label='Estimated', color='orange', edgecolor='white')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(factor_names, fontsize=12)
ax1.set_ylabel('Loading')
ax1.set_title('Factor Loadings: True vs Estimated')
ax1.axhline(0, color='black', linewidth=0.5)
ax1.legend()

# Pie: how much does each factor explain?
var_mkt_c = np.var(betas_hat[1] * mkt)
var_smb_c = np.var(betas_hat[2] * smb)
var_hml_c = np.var(betas_hat[3] * hml)
var_resid = np.var(stock_ret - y_hat)
var_t = var_mkt_c + var_smb_c + var_hml_c + var_resid

sizes = [var_mkt_c/var_t, var_smb_c/var_t, var_hml_c/var_t, var_resid/var_t]
pie_labels = [f'MKT\n{sizes[0]:.0%}', f'SMB\n{sizes[1]:.0%}', f'HML\n{sizes[2]:.0%}', f'Residual\n{sizes[3]:.0%}']
ax2.pie(sizes, labels=pie_labels, colors=['steelblue', 'orange', 'green', 'lightgrey'],
        autopct='', startangle=90, textprops={'fontsize': 11})
ax2.set_title(f'Variance Attribution  (R\u00b2 = {r2:.0%})')

plt.tight_layout()
plt.show()

print(f'Estimated loadings:  MKT={betas_hat[1]:.2f}  SMB={betas_hat[2]:.2f}  HML={betas_hat[3]:.2f}')
print(f'True loadings:       MKT={beta_mkt}  SMB={beta_smb}  HML={beta_hml}')
print(f'R-squared: {r2:.1%} — factors explain this much of the stock\'s variance')

### Beyond Three Factors

The industry has expanded well beyond Fama-French. Common additional factors:

- **Momentum (MOM)**: winners keep winning (short-term)
- **Quality (QMJ)**: profitable, stable companies outperform junk
- **Low Volatility**: boring stocks beat exciting ones risk-adjusted
- **Betting Against Beta (BAB)**: low-beta stocks outperform high-beta

Your risk system likely uses 5-10+ factors. The principle is the same: decompose returns to see what's factor-driven and what's genuine stock picking.

---
## 5. R-Squared — How Much Is Factor-Driven?

R-squared is the percentage of return variation that the factor model explains. It answers: **how much of what this PM does could you replicate with factor ETFs?**

$$R^2 = \frac{\text{variance explained by factors}}{\text{total variance of returns}}$$

| R² | What it means | Implication |
|-----|--------------|-------------|
| > 90% | Almost entirely factor-driven | Closet indexer — why pay active fees? |
| 70-90% | Mostly factor-driven | Typical for most long-only managers |
| 40-70% | Meaningful stock-specific bets | Active manager with genuine idiosyncratic bets |
| < 40% | Mostly stock-specific | True stock picker or unusual strategy |

In [ ]:
# === VISUAL: What different R-squared levels look like ===

np.random.seed(42)
n = 300
market_vis = np.random.normal(0.0004, 0.012, n)
dates_vis = pd.date_range('2023-01-01', periods=n, freq='B')

# Create stocks with different R² levels by controlling noise
noise_levels = [0.002, 0.008, 0.018, 0.035]  # low noise = high R²
r2_labels = []

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, noise_sd in zip(axes.flat, noise_levels):
    stock_v = 1.0 * market_vis + np.random.normal(0, noise_sd, n)
    r2_v = np.corrcoef(market_vis, stock_v)[0, 1] ** 2
    
    ax.scatter(market_vis * 100, stock_v * 100, alpha=0.2, s=8, color='steelblue')
    s_v, i_v = np.polyfit(market_vis, stock_v, 1)
    x_l = np.linspace(market_vis.min(), market_vis.max(), 50)
    ax.plot(x_l * 100, (s_v * x_l + i_v) * 100, 'r-', linewidth=1.5)
    ax.axhline(0, color='grey', linewidth=0.3)
    ax.axvline(0, color='grey', linewidth=0.3)
    ax.set_xlabel('Market (%)', fontsize=9)
    ax.set_ylabel('Stock (%)', fontsize=9)
    
    if r2_v > 0.9:
        verdict = 'Closet indexer'
    elif r2_v > 0.7:
        verdict = 'Mostly factor-driven'
    elif r2_v > 0.4:
        verdict = 'Active manager'
    else:
        verdict = 'True stock picker'
    
    ax.set_title(f'R\u00b2 = {r2_v:.0%} — {verdict}', fontsize=11)

plt.suptitle('R-Squared: How Tightly Returns Track the Market', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print('The R\u00b2 tension for LPs:')
print('  - They WANT low R\u00b2 (paying for alpha, not factor exposure)')
print('  - But low R\u00b2 = high tracking error = stomach-churning divergence from benchmarks')
print('  - This is why LPs panic when a true stock picker underperforms for 2 quarters')

---
## 6. Information Ratio — The Report Card for Active Managers

The IR answers: **how much alpha per unit of active risk?**

$$\text{IR} = \frac{\text{annualized alpha}}{\text{tracking error}}$$

Where tracking error = annualized std dev of the residuals (the day-to-day variability of alpha).

A PM who generates 3% alpha with 6% tracking error has IR = 0.5. A PM who generates 3% alpha with 15% tracking error has IR = 0.2. Same alpha, but the first delivers it **much** more consistently.

| IR | Rating | How rare |
|----|--------|----------|
| < 0.0 | Destroying value | ~40% of active managers |
| 0.0-0.3 | Below average | Common |
| 0.3-0.5 | Decent | Top quartile |
| 0.5-1.0 | Very good | Top decile |
| > 1.0 | Exceptional | Extremely rare sustained |

In [ ]:
# === TRY IT: Two PMs, same alpha, different consistency ===

np.random.seed(42)
n = 504  # 2 years
dates_ir = pd.date_range('2023-01-01', periods=n, freq='B')

alpha_ann = 0.04  # 4% annual alpha for both PMs
alpha_daily = alpha_ann / TRADING_DAYS

# PM A: consistent (low tracking error)
te_a = 0.005  # daily residual vol
residuals_a = np.random.normal(alpha_daily, te_a, n)
cumulative_a = pd.Series((1 + residuals_a).cumprod(), index=dates_ir)

# PM B: wild (high tracking error)
te_b = 0.018  # daily residual vol
residuals_b = np.random.normal(alpha_daily, te_b, n)
cumulative_b = pd.Series((1 + residuals_b).cumprod(), index=dates_ir)

ir_a = alpha_ann / (te_a * np.sqrt(TRADING_DAYS))
ir_b = alpha_ann / (te_b * np.sqrt(TRADING_DAYS))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(cumulative_a, linewidth=1, color='steelblue', label=f'PM A (IR={ir_a:.2f})')
ax1.plot(cumulative_b, linewidth=1, color='orange', label=f'PM B (IR={ir_b:.2f})')
ax1.axhline(1.0, color='grey', linewidth=0.5, linestyle='--')
ax1.set_title(f'Both PMs Have {alpha_ann:.0%} Alpha — But Look at the Ride', fontsize=11)
ax1.set_ylabel('Cumulative Residual Return')
ax1.legend(fontsize=11)

# Distribution of daily residuals
ax2.hist(residuals_a * 100, bins=60, alpha=0.6, color='steelblue', label=f'PM A (TE={te_a*np.sqrt(252)*100:.1f}%)', density=True)
ax2.hist(residuals_b * 100, bins=60, alpha=0.4, color='orange', label=f'PM B (TE={te_b*np.sqrt(252)*100:.1f}%)', density=True)
ax2.axvline(0, color='grey', linewidth=0.5)
ax2.set_xlabel('Daily Residual Return (%)')
ax2.set_title('Residual Distribution — Wider = More Active Risk', fontsize=11)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'PM A: Alpha={alpha_ann:.0%}, Tracking Error={te_a*np.sqrt(252):.1%}, IR={ir_a:.2f}')
print(f'PM B: Alpha={alpha_ann:.0%}, Tracking Error={te_b*np.sqrt(252):.1%}, IR={ir_b:.2f}')
print(f'\nSame alpha. PM A delivers it smoothly. PM B makes you seasick.')
print(f'LPs with short leashes need IR > 0.5 to justify the fee.')

### The Fundamental Law of Active Management

Grinold & Kahn decomposed IR into two pieces:

$$\text{IR} = \text{IC} \times \sqrt{\text{breadth}}$$

| Piece | What it means |
|-------|---------------|
| **IC** (Information Coefficient) | How good your predictions are — correlation between forecast and outcome. IC of 0.05 = slightly better than a coin flip. |
| **Breadth** | How many independent bets per year. Concentrated fund: ~30. Systematic fund: ~5000. |

This explains why systematic quant funds compete with (and often beat) fundamental stock pickers — tiny edge multiplied by massive breadth.

In [ ]:
# === TRY IT: Adjust IC and breadth — who wins? ===

ic_fundamental = 0.10   # <-- CHANGE: prediction quality per trade
breadth_fundamental = 30  # <-- CHANGE: trades per year

ic_quant = 0.02        # <-- CHANGE: much smaller edge per trade
breadth_quant = 5000   # <-- CHANGE: but way more trades

ir_fund = ic_fundamental * np.sqrt(breadth_fundamental)
ir_quant = ic_quant * np.sqrt(breadth_quant)

# Visualize the trade-off
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: IC vs Breadth space
breadths = np.logspace(1, 4, 200)
for target_ir, color, label in [(0.3, 'lightblue', 'IR=0.3 (below avg)'),
                                  (0.5, 'steelblue', 'IR=0.5 (decent)'),
                                  (1.0, 'navy', 'IR=1.0 (exceptional)')]:
    ic_needed = target_ir / np.sqrt(breadths)
    ax1.plot(breadths, ic_needed, color=color, linewidth=1.5, label=label)

ax1.scatter(breadth_fundamental, ic_fundamental, s=120, c='red', zorder=5, marker='*',
            label=f'Fundamental PM (IR={ir_fund:.2f})')
ax1.scatter(breadth_quant, ic_quant, s=120, c='green', zorder=5, marker='*',
            label=f'Quant Fund (IR={ir_quant:.2f})')
ax1.set_xscale('log')
ax1.set_xlabel('Breadth (trades/year)', fontsize=11)
ax1.set_ylabel('IC (prediction quality)', fontsize=11)
ax1.set_title('The Fundamental Law Trade-Off', fontsize=12)
ax1.legend(fontsize=9, loc='upper right')

# Right: bar comparison
categories = ['Fundamental PM', 'Quant Fund']
irs = [ir_fund, ir_quant]
bar_colors = ['red', 'green']
bars = ax2.bar(categories, irs, color=bar_colors, edgecolor='white', width=0.5)
for bar, ir_val in zip(bars, irs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'IR = {ir_val:.2f}', ha='center', fontweight='bold', fontsize=12)
ax2.axhline(0.5, color='grey', linewidth=0.8, linestyle='--', label='IR = 0.5 threshold')
ax2.set_ylabel('Information Ratio', fontsize=11)
ax2.set_title('IC \u00d7 \u221aBreadth = IR', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Fundamental PM:  IC={ic_fundamental} \u00d7 \u221a{breadth_fundamental} = IR {ir_fund:.2f}')
print(f'Quant fund:      IC={ic_quant} \u00d7 \u221a{breadth_quant} = IR {ir_quant:.2f}')
winner = 'Quant fund' if ir_quant > ir_fund else 'Fundamental PM'
print(f'\n{winner} wins. Not because it\'s smarter per trade, but because of breadth.')

---
## 7. Rolling Beta — Because Nothing Is Constant

Everything above assumes beta and factor loadings are fixed. They're not. They change constantly as market regimes shift, as PMs adjust positioning, and as stock character evolves.

**Rolling beta** computes beta over a sliding window (typically 63 trading days = ~3 months) and plots it over time. This reveals regime changes that a single number hides.

In [ ]:
# === DEMO: A stock whose beta shifts between regimes ===
#
# We simulate a stock that has beta=0.8 in calm markets
# but beta=1.6 during a crisis (correlations spike).

np.random.seed(42)
n = 756  # ~3 years
dates_rb = pd.date_range('2022-01-01', periods=n, freq='B')

# Market returns — normal most of the time, a crisis in the middle
market_rb = np.random.normal(0.0004, 0.01, n)
crisis_start, crisis_end = 250, 320  # ~3 month crisis
market_rb[crisis_start:crisis_end] = np.random.normal(-0.002, 0.025, crisis_end - crisis_start)

# Stock: beta changes during the crisis
true_beta = np.full(n, 0.8)
true_beta[crisis_start:crisis_end] = 1.6  # beta doubles in crisis

stock_rb = true_beta * market_rb + np.random.normal(0, 0.006, n)

# Compute rolling beta
window = 63
market_s = pd.Series(market_rb, index=dates_rb)
stock_s = pd.Series(stock_rb, index=dates_rb)
rolling_cov = stock_s.rolling(window).cov(market_s)
rolling_var = market_s.rolling(window).var()
rb = (rolling_cov / rolling_var).dropna()

# Static beta (full-period OLS)
static_beta = np.polyfit(market_rb, stock_rb, 1)[0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: market returns with crisis highlighted
ax1.plot(dates_rb, market_rb * 100, linewidth=0.5, color='steelblue')
ax1.axvspan(dates_rb[crisis_start], dates_rb[crisis_end-1], alpha=0.2, color='red', label='Crisis period')
ax1.set_ylabel('Market Return (%)')
ax1.set_title('Market Returns — Note the Crisis Period')
ax1.legend()

# Bottom: rolling beta vs static beta
ax2.plot(rb, linewidth=1, color='steelblue', label=f'{window}-day rolling beta')
ax2.axhline(static_beta, color='orange', linewidth=1.5, linestyle='--',
            label=f'Static beta = {static_beta:.2f} (hides regime shift)')
ax2.axhline(0.8, color='green', linewidth=0.8, linestyle=':', alpha=0.7, label='True calm beta = 0.80')
ax2.axhline(1.6, color='red', linewidth=0.8, linestyle=':', alpha=0.7, label='True crisis beta = 1.60')
ax2.axvspan(dates_rb[crisis_start], dates_rb[crisis_end-1], alpha=0.1, color='red')
ax2.set_ylabel('Beta')
ax2.set_title('Rolling Beta Reveals What Static Beta Hides')
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Static (full-period) beta: {static_beta:.2f}')
print(f'  Calm-period beta (true): 0.80')
print(f'  Crisis-period beta (true): 1.60')
print(f'\nThe static beta of {static_beta:.2f} looks fine — but it hides the fact')
print(f'that beta DOUBLED during the crisis. This is the scenario that breaks')
print(f'portfolios built on static assumptions.')

In [ ]:
# === EXERCISE: What happens if you make the crisis longer or more severe? ===
#
# Go back to the cell above and try:
#   1. Change crisis_end from 320 to 400 (longer crisis)
#   2. Change the crisis beta from 1.6 to 2.5 (more severe)
#   3. Change the rolling window from 63 to 21 (faster-reacting but noisier)
#
# Notice: shorter windows detect regime changes faster but are noisier.
# Longer windows are smoother but slower to react. This is a fundamental
# trade-off in all time-series analysis.

print('Go back to the cell above, change the parameters, and re-run!')

---
## 8. Reading a Factor Report — Red Flags and Good Signs

When you see factor model output on your risk report, here's what to look for:

### Red Flags

| Signal | What it might mean |
|--------|--------------------|
| Alpha t-stat < 2 | "Alpha" is probably noise, not skill |
| R² > 90% | Closet indexing — paying active fees for passive exposure |
| Negative alpha, significant | Actively destroying value after factor risk |
| Beta drifting sharply | PM taking unintended directional risk |
| Unexpected factor exposure | E.g., a "growth" fund with positive HML — they're buying value |

### Good Signs

| Signal | What it might mean |
|--------|--------------------|
| Alpha t-stat > 2 | Genuine stock-picking skill (or a very long lucky streak) |
| R² in 40-70% range | Active bets with factor awareness |
| IR > 0.5 | Consistent alpha relative to tracking error |
| Stable rolling beta | PM maintains discipline, exposure is intentional |

Let's build a mini factor report and practice reading it.

In [ ]:
# === BUILD A FACTOR REPORT — Diagnose these 4 synthetic portfolios ===

np.random.seed(42)
n = 756
mkt_r = np.random.normal(0.0004, 0.012, n)
smb_r = np.random.normal(0.0001, 0.005, n)
hml_r = np.random.normal(0.0001, 0.004, n)

portfolios = {
    'Closet Indexer': {'beta': 0.98, 'smb': -0.02, 'hml': 0.01, 'alpha': 0.00001, 'noise': 0.002},
    'True Alpha':     {'beta': 0.60, 'smb': -0.20, 'hml': -0.30, 'alpha': 0.0004, 'noise': 0.012},
    'Factor Loader':  {'beta': 1.30, 'smb': 0.50, 'hml': 0.60, 'alpha': -0.0001, 'noise': 0.008},
    'Market Neutral': {'beta': 0.05, 'smb': 0.00, 'hml': 0.00, 'alpha': 0.0003, 'noise': 0.005},
}

print(f'{"Portfolio":<18} {"Beta":>6} {"SMB":>6} {"HML":>6} {"Alpha(ann)":>11} {"R\u00b2":>6} {"IR":>6}  Diagnosis')
print('-' * 85)

for name, params in portfolios.items():
    ret = (params['alpha'] + params['beta'] * mkt_r + params['smb'] * smb_r
           + params['hml'] * hml_r + np.random.normal(0, params['noise'], n))
    
    # Regression
    X = np.column_stack([np.ones(n), mkt_r, smb_r, hml_r])
    b = np.linalg.lstsq(X, ret, rcond=None)[0]
    y_hat = X @ b
    resid = ret - y_hat
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((ret - ret.mean())**2)
    r2 = 1 - ss_res / ss_tot
    
    alpha_ann = b[0] * TRADING_DAYS
    te = np.std(resid) * np.sqrt(TRADING_DAYS)
    ir = alpha_ann / te if te > 0 else 0
    
    # Diagnosis
    if r2 > 0.90:
        diag = 'Closet indexer'
    elif abs(alpha_ann) > 0.03 and ir > 0.3:
        diag = 'Genuine alpha'
    elif alpha_ann < -0.02:
        diag = 'Destroying value'
    elif params['beta'] < 0.15 and r2 < 0.1:
        diag = 'Market-neutral'
    else:
        diag = 'Factor exposure, not alpha'
    
    print(f'{name:<18} {b[1]:>6.2f} {b[2]:>6.2f} {b[3]:>6.2f} {alpha_ann:>10.2%} {r2:>6.1%} {ir:>6.2f}  {diag}')

In [ ]:
# === EXERCISE: Create your own portfolio and diagnose it ===
#
# Fill in the parameters below to simulate a portfolio.
# Then look at the output and figure out: would you invest in this PM?

my_portfolio = {
    'beta': 1.0,      # <-- market sensitivity
    'smb': 0.0,       # <-- size tilt (+ small, - large)
    'hml': 0.0,       # <-- value tilt (+ value, - growth)
    'alpha': 0.0002,  # <-- daily alpha (0.0004 \u2248 10% annual)
    'noise': 0.008,   # <-- idiosyncratic vol
}

np.random.seed(123)
ret_my = (my_portfolio['alpha'] + my_portfolio['beta'] * mkt_r
          + my_portfolio['smb'] * smb_r + my_portfolio['hml'] * hml_r
          + np.random.normal(0, my_portfolio['noise'], n))

X_my = np.column_stack([np.ones(n), mkt_r, smb_r, hml_r])
b_my = np.linalg.lstsq(X_my, ret_my, rcond=None)[0]
y_hat_my = X_my @ b_my
resid_my = ret_my - y_hat_my
r2_my = 1 - np.sum(resid_my**2) / np.sum((ret_my - ret_my.mean())**2)
alpha_ann_my = b_my[0] * TRADING_DAYS
te_my = np.std(resid_my) * np.sqrt(TRADING_DAYS)
ir_my = alpha_ann_my / te_my if te_my > 0 else 0

print('Your Portfolio Report')
print('=' * 40)
print(f'  MKT loading: {b_my[1]:.3f}')
print(f'  SMB loading: {b_my[2]:.3f}')
print(f'  HML loading: {b_my[3]:.3f}')
print(f'  Alpha (ann): {alpha_ann_my:.2%}')
print(f'  R-squared:   {r2_my:.1%}')
print(f'  IR:          {ir_my:.2f}')
print()
print('Would you invest? What does each number tell you?')

---
## What to Explore Next

**In this repo:**
- `notebooks/factor_models_explained.ipynb` — runs the same analysis on **real market data** (AAPL, JPM, XOM)
- `modules/module_3_factor_models.py` — full Python implementation of everything in this guide
- `factor_dashboard.py` — CLI tool to run factor analysis on any ticker
- `modules/module_4_portfolio_optimisation.py` — the next step: construct portfolios that maximize alpha while controlling factor exposure

**Books:**
- **Grinold & Kahn**, *Active Portfolio Management* — the definitive reference on IR and the fundamental law
- **Andrew Ang**, *Asset Management* — bridges academic factor theory with how practitioners use it
- **Paleologo**, *Advanced Portfolio Management* — connects factor models to portfolio construction at institutional scale